In [1]:
import numpy as np
import numpy.linalg as linalg
import scipy.optimize as op
import scipy.interpolate
from scipy.integrate import quad
import pandas as pd
import time
from cobaya import run
import getdist.plots as gdplt
from getdist import MCSamples
import matplotlib.pyplot as plt
import os

c = 299792.458#km/s

# Carregando dados
## Dados CC

In [2]:
zHz, Hzi, errHz = np.genfromtxt('data/32CCdata.dat', comments='#', usecols=(0,1,2), unpack=True, delimiter=',')
zmod, imf, slib, sps, spsooo = np.genfromtxt('data/data_MM20.dat', comments='#', usecols=(0,1,2,3,4), unpack=True)

cov_mat_diag = np.zeros((len(zHz), len(zHz)), dtype='float64') 

for i in range(len(zHz)):
    cov_mat_diag[i,i] = errHz[i]**2#Erros estatísticos

#Erros sistemáticos
imf_intp = np.interp(zHz, zmod, imf)/100
spsooo_intp = np.interp(zHz, zmod, spsooo)/100

cov_mat_imf = np.zeros((len(zHz), len(zHz)), dtype='float64')
cov_mat_spsooo = np.zeros((len(zHz), len(zHz)), dtype='float64')

for i in range(len(zHz)):
    for j in range(len(zHz)):
        cov_mat_imf[i,j] = Hzi[i] * imf_intp[i] * Hzi[j] * imf_intp[j]
        cov_mat_spsooo[i,j] = Hzi[i] * spsooo_intp[i] * Hzi[j] * spsooo_intp[j]
        
#Sistemático+estatístico:
cov_mat = cov_mat_spsooo+cov_mat_imf+cov_mat_diag
inv_cov_mat = np.linalg.inv(cov_mat)

## Dados Pantheon+SH0ES

In [3]:
def build_data(datafile,SH0ES):
    """
    Run once at the start to load in the data vectors.
    
    Returns x, y where x is the independent variable (redshift in this case)
    and y is the Gaussian-distribured measured variable (magnitude in this case).
    """
    
    global origlen, wwPS, wwPPlus, cepheid_distance, is_calibrator, zHELPS, zHELPPlus

    print("Loading data from {}".format(datafile))
    data = pd.read_csv(datafile, sep=r'\s+')
    origlen = len(data)
    
    if SH0ES:
        ww = (data['zHD']>0.01) | (np.array(data['IS_CALIBRATOR'],dtype=bool))
        wwPS = ww
        zHELPS = data['zHEL'][ww]
    else:
        ww = (data['zHD']>0.01)
        wwPPlus = ww
        zHELPPlus = data['zHEL'][ww]

    zCMB = data['zHD'][ww] #use the vpec corrected redshift for zCMB 
    
    m_obs = data['m_b_corr'][ww]
    if SH0ES:
        is_calibrator = data['IS_CALIBRATOR'][ww]
        cepheid_distance = data['CEPH_DIST'][ww]

    return zCMB, m_obs

def build_covariance(covdatafile, SH0ES):
    """
    Run once at the start to build the covariance matrix for the data
    The file format for the covariance has the first line as an integer
    indicating the number of covariance elements, and the the subsequent
    lines being the elements.
    This function reads in the file and the nasty for loops trim down the covariance
    to match the only rows of data that are used for cosmology    
    """
    print("Loading covariance from {}".format(covdatafile))


    f = open(covdatafile)
    line = f.readline()
    if SH0ES:
        n = int(len(zCMBPS))
        ww = wwPS
    else:
        n = int(len(zCMBPPlus))
        ww = wwPPlus
    C = np.zeros((n,n))
    ii = -1
    jj = -1
    mine = 999
    maxe = -999
    for i in range(origlen):
        jj = -1
        if ww[i]:
            ii += 1
        for j in range(origlen):
            if ww[j]:
                jj += 1
            val = float(f.readline())
            if ww[i]:
                if ww[j]:
                    C[ii,jj] = val
    f.close()
    
    print('Done')

        # Return the covariance; the parent class knows to invert this
        # later to get the precision matrix that we need for the likelihood.
    return C

### Pantheon+SH0ES

In [4]:
zCMBPS, m_obsPS = build_data('data/Pantheon+SH0ES.dat',SH0ES=True)

covPS = build_covariance('data/Pantheon+SH0ES_STAT+SYS.cov', SH0ES=True)
invcovPS = linalg.inv(covPS)

Loading data from data/Pantheon+SH0ES.dat
Loading covariance from data/Pantheon+SH0ES_STAT+SYS.cov
Done


### Pantheon+

In [5]:
zCMBPPlus, m_obsPPlus = build_data('data/Pantheon+SH0ES.dat',SH0ES=False)

covPPlus = build_covariance('data/Pantheon+SH0ES_STAT+SYS.cov', SH0ES=False)
invcovPPlus = linalg.inv(covPPlus)

Loading data from data/Pantheon+SH0ES.dat
Loading covariance from data/Pantheon+SH0ES_STAT+SYS.cov
Done


# Modelos

In [6]:
#Theory points for Pantheon+ and Pantheon+&SH0ES
def extract_theory_points(theory_x, theory_y, pars, SH0ES):
    """
    Run once per parameter set to extract the mean vector that our
    data points are compared to.  For the Hubble flow set, we compare to the 
    cosmological model. For the calibrators we compare to the Cepheid distances.
    """
    M = pars[0]

    # Pull out theory mu and z from the block.
    #theory_x = block[self.x_section, self.x_name]
    #theory_y = block[self.y_section, self.y_name]

    # Interpolation function of theory so we can evaluate at redshifts of the data
    f = scipy.interpolate.interp1d(theory_x, theory_y)#, kind=self.kind)
    
    if SH0ES:
        theory_ynew = zCMBPS*np.nan
        #Here we use the Cepheid host distances as the "theory". 
        #This is the 2nd rung of the distance ladder and is what calibrates M
        theory_ynew[np.array(is_calibrator,dtype='bool')] = cepheid_distance[np.array(is_calibrator,dtype='bool')]
        
        # Actually do the interpolation at the data redshifts
        zcmb = zCMBPS[~np.array(is_calibrator,dtype='bool')]
        zhel = zHELPS[~np.array(is_calibrator,dtype='bool')]
    else:
        theory_ynew = zCMBPPlus*np.nan
        # Actually do the interpolation at the data redshifts
        zcmb = zCMBPPlus
        zhel = zHELPPlus
    
    fz = f(zcmb)
    if np.any(fz<=0):
        NegDist = True
        return 1
    if SH0ES:
        theory_ynew[~np.array(is_calibrator,dtype='bool')] = 5.0*np.log10((1.0+zcmb)*(1.0+zhel)*\
                                                                          np.atleast_1d(fz))+25.
    else:
        theory_ynew = 5.0*np.log10((1.0+zcmb)*(1.0+zhel)*np.atleast_1d(fz))+25.
                
    # Add the absolute supernova magnitude and return
    #M = block[names.supernova_params, "M"]
    return theory_ynew + M

def extract_theory_pointsHz(theory_x, theory_y, pars):
    theory_ynew = zHz*np.nan  

    f = scipy.interpolate.interp1d(theory_x, theory_y)#, kind=self.kind)
    theory_ynew = np.atleast_1d(f(zHz))
    return theory_ynew

#Points for theory interpolation
N = 3000
zt = np.linspace(0,2.5,N)

## P21

In [7]:
#H(z) obtido derivando Dc
def EzP21(z, theta):
    M, H0, q0, j0 = theta

    a1 = 1/H0
    a2 = (1 -2 * j0 + 2 * q0 + 3 * (q0 ** 2))/(6*H0*(1 + q0))
    b1 = (2 -j0 + 4 * q0 + 3 * (q0 ** 2))/(3*(1 + q0))

    hz = ((1 + b1 * z) ** 2) / (a1 + a2 * z * (2 + b1 * z))

    return hz/H0
    
#Distancia comovel adimensional
def DcModelP21(z, theta):
    M, H0, q0, j0 = theta
    
    a1 = 1/H0
    a2 = (1 -2 * j0 + 2 * q0 + 3 * (q0 ** 2))/(6*H0*(1 + q0))
    b1 = (2 -j0 + 4 * q0 + 3 * (q0 ** 2))/(3*(1 + q0))

    dc = (a1*z+a2*z**2)/(1+b1*z)

    return H0*dc

#Distancia comovel transversal in pc
def dmModelP21(z, theta):
    M, H0, q0, j0 = theta
    
    dH = c/H0
    wk = 0
    Dcm = DcModelP21(z, theta)
    if wk < 0:
        return dH*np.sin(np.sqrt(-wk)*Dcm)/np.sqrt(-wk)
    elif wk > 0:
        return dH*np.sinh(np.sqrt(wk)*Dcm)/np.sqrt(wk)
    return dH*Dcm

#distancia de luminosidade in pc
def dlModelP21(z,theta):
    return (1+z)*dmModelP21(z,theta)

def daModelP21(z,theta):
    return dmModelP21(z,theta)/(1+z)

def muModelP21(z,pars):
    return 5.*np.log10(dlModelP21(z,pars))+25

## P22

In [8]:
def EzP22(z, theta):
    M, H0, q0, j0, s0 = theta
    
    a1 = 1/H0
    a2 = ((-1 + 5 * j0 -3 * q0 + 6 * j0 * q0 -8 * (q0 ** 2) -6 * (q0 ** 3) + s0)) / (2*H0*((-1 + 2 * j0 -2 * q0 -3 * (q0 ** 2))))
    b1 = ((-2 + 7 * j0 -6 * q0 + 8 * j0 * q0 -13 * (q0 ** 2) -9 * (q0 ** 3) + s0)) /(2*(-1 + 2 * j0 -2 * q0 -3 * (q0 ** 2)))
    b2 = (2 -11 * j0 -4 * (j0 ** 2) + 8 * q0 -25 * j0 * q0 + 23 * (q0 ** 2) -6 * j0 * (q0 ** 2) + 30 * (q0 ** 3) + 9 * (q0 ** 4) -3 * s0
          -3 * q0 *s0) / (12 -24 * j0 + 24 * q0 + 36 * (q0 ** 2))

    hz = ((1 + b1 * z + b2 * (z ** 2)) ** 2) / (a1 -(a1 * b2 * (z ** 2)) + a2 *z * (2 + b1 * z))

    return hz/H0

def DcModelP22(z, theta):
    M, H0, q0, j0, s0 = theta
    
    a1 = 1/H0
    a2 = ((-1 + 5 * j0 -3 * q0 + 6 * j0 * q0 -8 * (q0 ** 2) -6 * (q0 ** 3) + s0)) / (2*H0*((-1 + 2 * j0 -2 * q0 -3 * (q0 ** 2))))
    b1 = ((-2 + 7 * j0 -6 * q0 + 8 * j0 * q0 -13 * (q0 ** 2) -9 * (q0 ** 3) + s0)) /(2*(-1 + 2 * j0 -2 * q0 -3 * (q0 ** 2)))
    b2 = (2 -11 * j0 -4 * (j0 ** 2) + 8 * q0 -25 * j0 * q0 + 23 * (q0 ** 2) -6 * j0 * (q0 ** 2) + 30 * (q0 ** 3) + 9 * (q0 ** 4) -3 * s0
          -3 * q0 *s0) / (12 -24 * j0 + 24 * q0 + 36 * (q0 ** 2))

    dc = (a1*z + a2*z**2)/(1+b1*z+b2*z**2)
    
    return dc

#Distancia comovel transversal in pc
def dmModelP22(z, theta):
    M, H0, q0, j0, s0 = theta
    
    dH = c/H0
    wk = 0
    Dcm = DcModelP22(z, theta)
    if wk < 0:
        return dH*np.sin(np.sqrt(-wk)*Dcm)/np.sqrt(-wk)
    elif wk > 0:
        return dH*np.sinh(np.sqrt(wk)*Dcm)/np.sqrt(wk)
    return dH*Dcm

#distancia de luminosidade in pc
def dlModelP22(z,theta):
    return (1+z)*dmModelP22(z,theta)

def daModelP22(z,theta):
    return dmModelP22(z,theta)/(1+z)

def muModelP22(z,pars):
    return 5.*np.log10(dlModelP22(z,pars))+25

## P31

In [9]:
def EzP31(z, theta):
    M, H0, q0, j0, s0 = theta

    a1 = 1/H0
    a2 = ((-2 + 7 * j0 -6 * q0 + 8 * j0 * q0 -13 * (q0 ** 2) -9 * (q0 **3) + s0))/(4*H0*(-2 + j0 -4 * q0 -3 * (q0 ** 2)))
    a3 = ((2 -11 * j0 -4 * (j0 ** 2) + 8 * q0 -25 * j0 * q0 + 23 * (q0 ** 2) -6 * j0 * (q0 ** 2) + 30 * (q0 ** 3) + 9 * (q0 **4)
          -3 * s0 -3 * q0 * s0)) /(24*H0*(-2 + j0 -4 * q0 -3 * (q0 ** 2)))
    b1 = ((-6 + 9 * j0 -18 * q0 + 10 * j0 * q0 -27 * (q0 ** 2) -15 *(q0 ** 3) + s0)) / (4*(-2 + j0 -4 * q0 -3 * (q0 ** 2)))

    hz = ((1 + b1 * z) ** 2) / (a1 + z * (a2 * (2 + b1 * z) + a3 * z * (3 + 2 *b1 * z)))

    return hz/H0

def DcModelP31(z, theta):
    M, H0, q0, j0, s0 = theta

    a1 = 1/H0
    a2 = ((-2 + 7 * j0 -6 * q0 + 8 * j0 * q0 -13 * (q0 ** 2) -9 * (q0 **3) + s0))/(4*H0*(-2 + j0 -4 * q0 -3 * (q0 ** 2)))
    a3 = ((2 -11 * j0 -4 * (j0 ** 2) + 8 * q0 -25 * j0 * q0 + 23 * (q0 ** 2) -6 * j0 * (q0 ** 2) + 30 * (q0 ** 3) + 9 * (q0 **4)
          -3 * s0 -3 * q0 * s0)) /(24*H0*(-2 + j0 -4 * q0 -3 * (q0 ** 2)))
    b1 = ((-6 + 9 * j0 -18 * q0 + 10 * j0 * q0 -27 * (q0 ** 2) -15 *(q0 ** 3) + s0)) / (4*(-2 + j0 -4 * q0 -3 * (q0 ** 2)))

    dc = (a1*z + a2*z**2 + a3*z**3)/(1 + b1*z)

    return c*dc

#Distancia comovel transversal in pc
def dmModelP31(z, theta):
    M, H0, q0, j0, s0= theta
    
    dH = c/H0
    wk = 0
    Dcm = DcModelP31(z, theta)
    if wk < 0:
        return dH*np.sin(np.sqrt(-wk)*Dcm)/np.sqrt(-wk)
    elif wk > 0:
        return dH*np.sinh(np.sqrt(wk)*Dcm)/np.sqrt(wk)
    return dH*Dcm

#distancia de luminosidade in pc
def dlModelP31(z,theta):
    return (1+z)*dmModelP31(z,theta)

def daModelP31(z,theta):
    return dmModelP31(z,theta)/(1+z)

def muModelP31(z,pars):
    return 5.*np.log10(dlModelP31(z,pars))+25

# Análise estatística
## Funções estatísticas

In [10]:
def find_bestfit(lnlike, parnames, par_ml):#,data):
    t1 = time.time()
    ndim = len(par_ml)
    chi2 = lambda *args: -2 * lnlike(*args)
    result = op.minimize(chi2, par_ml)#, args=data)
    if not result['success']:
        result = op.minimize(chi2, par_ml, method='Nelder-Mead',options={'maxiter': 10000})#, args=data
    par_ml = result["x"]
    print('Maximum likelihood result:')
    for i in range(ndim):
        print(parnames[i],' = ',par_ml[i])
    print('chi2min =',result['fun'])
    #ndof = len(data[0])-ndim
    #print('chi2red =',result['fun']/ndof)
    #print(result)
    t2 = time.time()
    print("tempo total: {0:5.3f} seg".format(t2-t1))
    return result

def run_cobaya(info, info_post):
    print("----- INICIANDO SAMPLER -----")
    t1 = time.time()

    # Roda o sampler
    updated_info, sampler = run(info)

    # Retira o início 
    print("----- RETIRANDO O INICIO -----")
    updated_info_post, sampler_post = run(info_post)

    print(f"Tempo de execução: {time.time()-t1} s")

    return sampler, sampler_post

def MCResult_cobaya(sampler):
    gdsamples = sampler.products()["sample"][0].to_getdist()
    stats = gdsamples.getMargeStats()

    for par in stats.names[:gdsamples.samples.shape[1]-2]:
        par_stats = stats.parWithName(par.name)
        mean = par_stats.mean

        # Erros de 1-sigma (68% CL)
        lower_1s = par_stats.limits[0].lower
        upper_1s = par_stats.limits[0].upper
        err_plus_1s = upper_1s - mean
        err_minus_1s = mean - lower_1s

        # Erros de 2-sigma (95% CL)
        lower_2s = par_stats.limits[1].lower
        upper_2s = par_stats.limits[1].upper
        err_plus_2s = upper_2s - mean
        err_minus_2s = mean - lower_2s
            
        label = f"{par.name}"
        
            
        print(f"{label} = {mean:.4f} +{err_plus_1s:.4f} +{err_plus_2s:.4f} -{err_plus_1s:.4f} -{err_plus_2s:.4f}") 
    
def plot_getdist(samples, params, legends, width=8, fill=False):
    gsamples = []
    for i in samples:
        gsamples.append(i.products()["sample"][0].to_getdist())

    g = gdplt.getSubplotPlotter(width_inch=width)
    g.triangle_plot(gsamples, params, filled=fill, legend_labels=legends)
    plt.show()

## Estatística dos modelos
### P21

In [11]:
def getpar_PSHz_21(theta):
    M, H0, q0, j0 = theta
    return lnprobPSHz_21(M, H0, q0, j0)

def getpar_PPHz_21(theta):
    M, H0, q0, j0 = theta
    return lnprobPPHz_21(M, H0, q0, j0)

def chi2PS_21(theta):#Pantheon+SH0ES
    zsni = zCMBPS
    Ez2i = EzP21(zsni,theta)
    if np.any(Ez2i <= 0):
        return np.inf

    dAt = daModelP21(zt,theta)
    NegDist = False
    mmod = extract_theory_points(zt, dAt, theta, SH0ES=True)
    
    if NegDist:
        return np.inf
    dm = mmod-m_obsPS
    
    return np.dot(np.dot(dm.T,invcovPS),dm)
        
def chi2PP_21(theta):#Pantheon+
    zsni = zCMBPPlus
    Ez2i = EzP21(zsni,theta)
    if np.any(Ez2i <= 0):
        return np.inf

    dAt = daModelP21(zt,theta)
    NegDist = False
    mmod = extract_theory_points(zt, dAt, theta,SH0ES=False) #5.*np.log10((1+zsni)*(1+zheli)*dAmodi/10)#(dlmodi/10)
    if NegDist:
        return np.inf
    dmu = mmod-m_obsPPlus
    SA  = np.sum(invcovPPlus)
    Sr  = np.sum(np.dot(dmu.T,invcovPPlus))#np.sum(dmu*invcov)
    Srr = np.dot(np.dot(dmu.T,invcovPPlus),dmu)#np.sum(dmu*dmu*invcov)
    return Srr - Sr**2./SA
        
def chi2Hz_21(theta):#CC Hz
    M, H0, q0, j0 = theta
        
    Ezt = EzP21(zt,theta)
    if np.any(Ezt<0):
        return np.inf
    Hzt = H0*Ezt#Hz(zt,pars)
    Hzm = extract_theory_pointsHz(zt, Hzt, theta)
   
    dh = Hzm-Hzi
    
    return np.dot(np.dot(dh.T,inv_cov_mat),dh)

def lnlikePS_21(theta):
    return -0.5*chi2PS_21(theta)

def lnlikePP_21(theta):
    return -0.5*chi2PP_21(theta)

def lnlikeHz_21(theta):
    return -0.5*chi2Hz_21(theta)

def lnprobPSHz_21(M, h0, q0, j0):
    theta = [M, h0, q0, j0]
    return lnlikePS_21(theta)+lnlikeHz_21(theta)
    
def lnprobPPHz_21(M, h0, q0, j0):
    theta = [M, h0, q0, j0]
    return lnlikePP_21(theta)+lnlikeHz_21(theta)

### P22

In [12]:
def getpar_PSHz_22(theta):
    M, h0, q0, j0, s0 = theta
    return lnprobPSHz_22(M, h0, q0, j0, s0)

def getpar_PPHz_22(theta):
    M, h0, q0, j0, s0 = theta
    return lnprobPPHz_22(M, h0, q0, j0, s0)

def chi2PS_22(theta):#Pantheon+SH0ES
    zsni = zCMBPS
    Ez2i = EzP22(zsni,theta)
    if np.any(Ez2i <= 0):
        return np.inf

    dAt = daModelP22(zt,theta)
    NegDist = False
    mmod = extract_theory_points(zt, dAt, theta, SH0ES=True)
    if NegDist:
        return np.inf
    dm = mmod-m_obsPS
    
    return np.dot(np.dot(dm.T,invcovPS),dm)
        
def chi2PP_22(theta):#Pantheon+
    zsni = zCMBPPlus
    Ez2i = EzP22(zsni,theta)
    if np.any(Ez2i <= 0):
        return np.inf

    dAt = daModelP22(zt,theta)
    NegDist = False
    mmod = extract_theory_points(zt, dAt, theta,SH0ES=False) #5.*np.log10((1+zsni)*(1+zheli)*dAmodi/10)#(dlmodi/10)
    if NegDist:
        return np.inf
    dmu = mmod-m_obsPPlus
    SA  = np.sum(invcovPPlus)
    Sr  = np.sum(np.dot(dmu.T,invcovPPlus))#np.sum(dmu*invcov)
    Srr = np.dot(np.dot(dmu.T,invcovPPlus),dmu)#np.sum(dmu*dmu*invcov)
    return Srr - Sr**2./SA
        
def chi2Hz_22(theta):#CC Hz
    M, H0, q0, j0, s0 = theta
        
    Ezt = EzP22(zt,theta)
    if np.any(Ezt<0):
        return np.inf
    Hzt = H0*Ezt#Hz(zt,pars)
    Hzm = extract_theory_pointsHz(zt, Hzt, theta)
   
    dh = Hzm-Hzi
    
    return np.dot(np.dot(dh.T,inv_cov_mat),dh)

def lnlikePS_22(theta):
    return -0.5*chi2PS_22(theta)

def lnlikePP_22(theta):
    return -0.5*chi2PP_22(theta)

def lnlikeHz_22(theta):
    return -0.5*chi2Hz_22(theta)

def lnprobPSHz_22(M, h0, q0, j0, s0):
    theta = [M, h0, q0, j0, s0]
    return lnlikePS_22(theta)+lnlikeHz_22(theta)
    
def lnprobPPHz_22(M, h0, q0, j0, s0):
    theta = [M, h0, q0, j0, s0]
    return lnlikePP_22(theta)+lnlikeHz_22(theta)

### P31

In [13]:
def getpar_PSHz_31(theta):
    M, h0, q0, j0, s0 = theta
    return lnprobPSHz_31(M, h0, q0, j0, s0)

def getpar_PPHz_31(theta):
    M, h0, q0, j0, s0 = theta
    return lnprobPPHz_31(M, h0, q0, j0, s0)

def chi2PS_31(theta):#Pantheon+SH0ES
    zsni = zCMBPS
    Ez2i = EzP31(zsni,theta)
    if np.any(Ez2i <= 0):
        return np.inf

    dAt = daModelP31(zt,theta)
    NegDist = False
    mmod = extract_theory_points(zt, dAt, theta, SH0ES=True)
    if NegDist:
        return np.inf
    dm = mmod-m_obsPS
    
    return np.dot(np.dot(dm.T,invcovPS),dm)
        
def chi2PP_31(theta):#Pantheon+
    zsni = zCMBPPlus
    Ez2i = EzP31(zsni,theta)
    if np.any(Ez2i <= 0):
        return np.inf

    dAt = daModelP31(zt,theta)
    NegDist = False
    mmod = extract_theory_points(zt, dAt, theta,SH0ES=False) #5.*np.log10((1+zsni)*(1+zheli)*dAmodi/10)#(dlmodi/10)
    if NegDist:
        return np.inf
    dmu = mmod-m_obsPPlus
    SA  = np.sum(invcovPPlus)
    Sr  = np.sum(np.dot(dmu.T,invcovPPlus))#np.sum(dmu*invcov)
    Srr = np.dot(np.dot(dmu.T,invcovPPlus),dmu)#np.sum(dmu*dmu*invcov)
    return Srr - Sr**2./SA
        
def chi2Hz_31(theta):#CC Hz
    M, H0, q0, j0, s0 = theta
        
    Ezt = EzP31(zt,theta)
    if np.any(Ezt<0):
        return np.inf
    Hzt = H0*Ezt#Hz(zt,pars)
    Hzm = extract_theory_pointsHz(zt, Hzt, theta)
   
    dh = Hzm-Hzi
    
    return np.dot(np.dot(dh.T,inv_cov_mat),dh)

def lnlikePS_31(theta):
    return -0.5*chi2PS_31(theta)

def lnlikePP_31(theta):
    return -0.5*chi2PP_31(theta)

def lnlikeHz_31(theta):
    return -0.5*chi2Hz_31(theta)

def lnprobPSHz_31(M, h0, q0, j0, s0):
    theta = [M, h0, q0, j0, s0]
    return lnlikePS_31(theta)+lnlikeHz_31(theta)
    
def lnprobPPHz_31(M, h0, q0, j0, s0):
    theta = [M, h0, q0, j0, s0]
    return lnlikePP_31(theta)+lnlikeHz_31(theta)

## Análise dos modelos
### P21

In [14]:
parlabels_21 = ["M","H_0",'q_0','j_0']
parlabtex_21 = ["$M$","$H_0$",'$q_0$','$j_0$']
parnames_21 = ['M','H0','q0','j0']
par0_21=[-19.2, 73.03398760350208, -0.5280171275112104, 1]
ndim_21 = len(par0_21)

#### Pantheon+CC

In [16]:
par_mlHzPP_21 = par0_21
resultHzPP_21 = find_bestfit(getpar_PPHz_21, parnames_21, par_mlHzPP_21)
par_mlHzPP_21=resultHzPP_21['x']
print('\n',par_mlHzPP_21,'\n')
print(resultHzPP_21,'\n')

Maximum likelihood result:
M  =  -19.2
H0  =  73.03398760350208
q0  =  -0.5280171275112104
j0  =  1.0
chi2min = inf
tempo total: 43.455 seg

 [-19.2         73.0339876   -0.52801713   1.        ] 

       message: Maximum number of iterations has been exceeded.
       success: False
        status: 2
           fun: inf
             x: [-1.920e+01  7.303e+01 -5.280e-01  1.000e+00]
           nit: 10000
          nfev: 59999
 final_simplex: (array([[-1.920e+01,  7.303e+01, -5.280e-01,  1.000e+00],
                       [-1.920e+01,  7.303e+01, -5.280e-01,  1.000e+00],
                       ...,
                       [-1.920e+01,  7.303e+01, -5.280e-01,  1.000e+00],
                       [-1.920e+01,  7.303e+01, -5.280e-01,  1.000e+00]],
                      shape=(5, 4)), array([       inf,        inf,        inf,        inf,
                              inf])) 



In [18]:
info_PPHz21 = {
    "likelihood": {
        "lnlikehz": lnprobPPHz_21
    },

    "params": {
        "M": {"prior": {"min": -19.5, "max": 19.5}, "proposal": par_mlHzPP_21[0], "latex": "M"},
        "h0": {"prior": {"min": 40.0, "max": 100.0}, "proposal": par_mlHzPP_21[1], "latex": r"H_{0}"},
        "q0": {"prior": {"min": -2, "max": 2}, "proposal": par_mlHzPP_21[2], "latex": r"q_{0}"},
        "j0": {"prior": {"min": -5, "max": 5}, "proposal": par_mlHzPP_21[3], "latex": r"j_{0}"}
    },

    "sampler": {
        "polychord": {
            "nlive": 250
        }
    },

    "output": "chains/P21/PP/PPHz_21"
}

info_post_PPHz21 = {
    "output": "chains/P21/PP/PPHz_21",
    "post": {
        "skip_samples": 0.3,
        "suffix": "_post"
    }
}

In [22]:
_, sampler_PPHz_21 = run_cobaya(info_PPHz21, info_post_PPHz21)

----- INICIANDO SAMPLER -----
[output] Output to be read-from/written-into folder 'chains/P21/PP', with prefix 'PPHz_21'
[output] Found existing info files with the requested output prefix: 'chains/P21/PP/PPHz_21'
[lnlikehz] Initialized external likelihood.
[polychord] `pypolychord` module loaded successfully from /home/nirk20/jupyter_venv/lib/python3.12/site-packages/pypolychord
[polychord] Storing raw PolyChord output in 'chains/P21/PP/PPHz_21_polychord_raw'.
[model] Measuring speeds... (this may take a few seconds)
[model] Setting measured speeds (per sec): {lnlikehz: 58.3}
[polychord] Parameter blocks and their oversampling factors:
[polychord] * 1 : ['M', 'h0', 'q0', 'j0']
[polychord] Calling PolyChord...
PolyChord: MPI is already initilised, not initialising, and will not finalize

PolyChord: Next Generation Nested Sampling
copyright: Will Handley, Mike Hobson & Anthony Lasenby
  version: 1.22.1
  release: 10th Jan 2024
    email: wh260@mrao.cam.ac.uk

Run Settings
nlive    :    

#### Pantheon+SH0ES+CC

In [19]:
par_mlHzPS_21 = par0_21
resultHzPS_21 = find_bestfit(getpar_PSHz_21, parnames_21, par_mlHzPS_21)
par_mlHzPS_21=resultHzPS_21['x']
print('\n',par_mlHzPS_21,'\n')
print(resultHzPS_21,'\n')

Maximum likelihood result:
M  =  -19.2
H0  =  73.03398760350208
q0  =  -0.5280171275112104
j0  =  1.0
chi2min = inf
tempo total: 42.630 seg

 [-19.2         73.0339876   -0.52801713   1.        ] 

       message: Maximum number of iterations has been exceeded.
       success: False
        status: 2
           fun: inf
             x: [-1.920e+01  7.303e+01 -5.280e-01  1.000e+00]
           nit: 10000
          nfev: 59999
 final_simplex: (array([[-1.920e+01,  7.303e+01, -5.280e-01,  1.000e+00],
                       [-1.920e+01,  7.303e+01, -5.280e-01,  1.000e+00],
                       ...,
                       [-1.920e+01,  7.303e+01, -5.280e-01,  1.000e+00],
                       [-1.920e+01,  7.303e+01, -5.280e-01,  1.000e+00]],
                      shape=(5, 4)), array([       inf,        inf,        inf,        inf,
                              inf])) 



In [22]:
info_PSHz21 = {
    "likelihood": {
        "lnlikehz": lnprobPSHz_21
    },

    "params": {
        "M": {"prior": {"min": -19.5, "max": 19.5}, "proposal": par_mlHzPS_21[0], "latex": "M"},
        "h0": {"prior": {"min": 40.0, "max": 100.0}, "proposal": par_mlHzPS_21[1], "latex": r"H_{0}"},
        "q0": {"prior": {"min": -2, "max": 0}, "proposal": par_mlHzPS_21[2], "latex": r"q_{0}"},
        "j0": {"prior": {"min": -5, "max": 5}, "proposal": par_mlHzPS_21[3], "latex": r"j_{0}"}
    },

    "sampler": {
        "polychord": {
            "nlive": 250
        }
    },

    "output": "chains/P21/PS/PSHz_21"
}

info_post_PSHz21 = {
    "output": "chains/P21/PS/PSHz_21",
    "post": {
        "skip_samples": 0.3,
        "suffix": "_post"
    }
}

In [23]:
_, sampler_PSHz_21 = run_cobaya(info_PSHz21, info_post_PSHz21)

----- INICIANDO SAMPLER -----
[output] Output to be read-from/written-into folder 'chains/P21/PS', with prefix 'PSHz_21'
[lnlikehz] Initialized external likelihood.
[polychord] `pypolychord` module loaded successfully from /home/nirk20/jupyter_venv/lib/python3.12/site-packages/pypolychord
[polychord] Storing raw PolyChord output in 'chains/P21/PS/PSHz_21_polychord_raw'.
[model] Measuring speeds... (this may take a few seconds)
[model] Setting measured speeds (per sec): {lnlikehz: 131.0}
[polychord] Parameter blocks and their oversampling factors:
[polychord] * 1 : ['M', 'h0', 'q0', 'j0']
[polychord] Calling PolyChord...
PolyChord: MPI is already initilised, not initialising, and will not finalize

PolyChord: Next Generation Nested Sampling
copyright: Will Handley, Mike Hobson & Anthony Lasenby
  version: 1.22.1
  release: 10th Jan 2024
    email: wh260@mrao.cam.ac.uk

Run Settings
nlive    :     250
nDims    :       4
nDerived :       2
Doing Clustering
Synchronous parallelisation
Gene

### P22

In [14]:
parlabels_22 = ["M","H_0",'q_0','j_0', 's_0']
parlabtex_22 = ["$M$","$H_0$",'$q_0$','$j_0$', '$s_0$']
parnames_22 = ['M','H0','q0','j0', 's0']
par0_22=[-19.2, 73.03398760350208, -0.5280171275112104, 1, -0.35]
ndim_22 = len(par0_22)

#### Pantheon+CC

In [68]:
par_mlHzPP_22 = par0_22
resultHzPP_22 = find_bestfit(getpar_PPHz_22, parnames_22, par_mlHzPP_22)
par_mlHzPP_22 = resultHzPP_22['x']
print('\n',par_mlHzPP_22,'\n')
print(resultHzPP_22,'\n')

Maximum likelihood result:
M  =  -2.6773366368543225
H0  =  67.77818713859679
q0  =  -0.48067105840284835
j0  =  0.9644912638887624
s0  =  0.40595788013237727
chi2min = 1417.274237609817
tempo total: 9.723 seg

 [-2.67733664 67.77818714 -0.48067106  0.96449126  0.40595788] 

       message: Optimization terminated successfully.
       success: True
        status: 0
           fun: 1417.274237609817
             x: [-2.677e+00  6.778e+01 -4.807e-01  9.645e-01  4.060e-01]
           nit: 212
          nfev: 377
 final_simplex: (array([[-2.677e+00,  6.778e+01, ...,  9.645e-01,
                         4.060e-01],
                       [-2.677e+00,  6.778e+01, ...,  9.645e-01,
                         4.060e-01],
                       ...,
                       [-2.677e+00,  6.778e+01, ...,  9.645e-01,
                         4.060e-01],
                       [-2.677e+00,  6.778e+01, ...,  9.645e-01,
                         4.060e-01]], shape=(6, 5)), array([ 1.417e+03,  1.417e+03, 

In [69]:
info_PPHz22 = {
    "likelihood": {
        "lnlikehz": lnprobPPHz_22
    },

    "params": {
        "M": {"prior": {"min": -19.5, "max": 19.5}, "proposal": par_mlHzPP_22[0], "latex": "M"},
        "h0": {"prior": {"min": 40.0, "max": 100.0}, "proposal": par_mlHzPP_22[1], "latex": r"H_{0}"},
        "q0": {"prior": {"min": 0, "max": 2}, "proposal": par_mlHzPP_22[2], "latex": r"q_{0}"},
        "j0": {"prior": {"min": -5, "max": 5}, "proposal": par_mlHzPP_22[3], "latex": r"j_{0}"},
        "s0": {"prior": {"min": -4, "max": 4}, "proposal": par_mlHzPP_22[4], "latex": r"s_{0}"}
    },

    "sampler": {
        "polychord": {
            "nlive":300
        }
    },

    "output": "chains/P22/PP/PPHz_22"
}

info_post_PPHz22 = {
    "output": "chains/P22/PP/PPHz_22",
    "post": {
        "skip_samples": 0.3,
        "suffix": "_post"
    }
}

In [70]:
_, sampler_PPHz_22 = run_cobaya(info_PPHz22, info_post_PPHz22)

----- INICIANDO SAMPLER -----
[output] Output to be read-from/written-into folder 'chains/P22/PP', with prefix 'PPHz_22'
[lnlikehz] Initialized external likelihood.
[polychord] `pypolychord` module loaded successfully from /home/nirk20/jupyter_venv/lib/python3.12/site-packages/pypolychord
[polychord] Storing raw PolyChord output in 'chains/P22/PP/PPHz_22_polychord_raw'.
[model] Measuring speeds... (this may take a few seconds)
[model] Setting measured speeds (per sec): {lnlikehz: 123.0}
[polychord] Parameter blocks and their oversampling factors:
[polychord] * 1 : ['M', 'h0', 'q0', 'j0', 's0']
[polychord] Calling PolyChord...
PolyChord: MPI is already initilised, not initialising, and will not finalize

PolyChord: Next Generation Nested Sampling
copyright: Will Handley, Mike Hobson & Anthony Lasenby
  version: 1.22.1
  release: 10th Jan 2024
    email: wh260@mrao.cam.ac.uk

Run Settings
nlive    :     300
nDims    :       5
nDerived :       2
Doing Clustering
Synchronous parallelisatio

#### Pantheon+SH0ES+CC

In [19]:
par_mlHzPS_22 = par0_22
resultHzPS_22 = find_bestfit(getpar_PSHz_22, parnames_22, par_mlHzPS_22)
par_mlHzPS_22 = resultHzPS_22['x']
print('\n',par_mlHzPS_22,'\n')
print(resultHzPS_22,'\n')

/home/nirk20/jupyter_venv/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:596: RuntimeWarning: invalid value encountered in subtract
  df = fun(x1) - f0


Maximum likelihood result:
M  =  -16.31415255626306
H0  =  16.698506393861372
q0  =  -0.6364690862763576
j0  =  1.6390812142067037
s0  =  -0.9592556490773507
chi2min = 11713.093699751356
tempo total: 9.118 seg

 [-16.31415256  16.69850639  -0.63646909   1.63908121  -0.95925565] 

       message: Optimization terminated successfully.
       success: True
        status: 0
           fun: 11713.093699751356
             x: [-1.631e+01  1.670e+01 -6.365e-01  1.639e+00 -9.593e-01]
           nit: 1023
          nfev: 1742
 final_simplex: (array([[-1.631e+01,  1.670e+01, ...,  1.639e+00,
                        -9.593e-01],
                       [-1.631e+01,  1.670e+01, ...,  1.639e+00,
                        -9.593e-01],
                       ...,
                       [-1.631e+01,  1.670e+01, ...,  1.639e+00,
                        -9.593e-01],
                       [-1.631e+01,  1.670e+01, ...,  1.639e+00,
                        -9.593e-01]], shape=(6, 5)), array([ 1.171e+04,  1.1

In [20]:
info_PSHz22 = {
    "likelihood": {
        "lnlikehz": lnprobPSHz_22
    },

    "params": {
        "M": {"prior": {"min": -19.5, "max": 19.5}, "proposal": par_mlHzPS_22[0], "latex": "M"},
        "h0": {"prior": {"min": 40.0, "max": 100.0}, "proposal": par_mlHzPS_22[1], "latex": r"H_{0}"},
        "q0": {"prior": {"min": -2, "max": 0}, "proposal": par_mlHzPS_22[2], "latex": r"q_{0}"},
        "j0": {"prior": {"min": -5, "max": 5}, "proposal": par_mlHzPS_22[3], "latex": r"j_{0}"},
        "s0": {"prior": {"min": -4, "max": 4}, "proposal": par_mlHzPS_22[4], "latex": r"s_{0}"}
    },

    "sampler": {
        "polychord": {
            "nlive":300
        }
    },

    "output": "chains/P22/PS/PSHz_22"
}

info_post_PSHz22 = {
    "output": "chains/P22/PS/PSHz_22",
    "post": {
        "skip_samples": 0.3,
        "suffix": "_post"
    }
}

In [21]:
_, sampler_PSHz_22 = run_cobaya(info_PSHz22, info_post_PSHz22)

----- INICIANDO SAMPLER -----
[output] Output to be read-from/written-into folder 'chains/P22/PS', with prefix 'PSHz_22'
[lnlikehz] Initialized external likelihood.
[polychord] `pypolychord` module loaded successfully from /home/nirk20/jupyter_venv/lib/python3.12/site-packages/pypolychord
[polychord] Storing raw PolyChord output in 'chains/P22/PS/PSHz_22_polychord_raw'.
[model] Measuring speeds... (this may take a few seconds)
[model] Setting measured speeds (per sec): {lnlikehz: 198.0}
[polychord] Parameter blocks and their oversampling factors:
[polychord] * 1 : ['M', 'h0', 'q0', 'j0', 's0']
[polychord] Calling PolyChord...
PolyChord: MPI is already initilised, not initialising, and will not finalize

PolyChord: Next Generation Nested Sampling
copyright: Will Handley, Mike Hobson & Anthony Lasenby
  version: 1.22.1
  release: 10th Jan 2024
    email: wh260@mrao.cam.ac.uk

Run Settings
nlive    :     300
nDims    :       5
nDerived :       2
Doing Clustering
Synchronous parallelisatio

### P31

#### Pantheon+CC

In [15]:
par_mlHzPP_31 = par0_22
resultHzPP_31 = find_bestfit(getpar_PPHz_31, parnames_22, par_mlHzPP_31)
par_mlHzPP_31 = resultHzPP_31['x']
print('\n',par_mlHzPP_31,'\n')
print(resultHzPP_31,'\n')

/home/nirk20/jupyter_venv/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:596: RuntimeWarning: invalid value encountered in subtract
  df = fun(x1) - f0
/home/nirk20/jupyter_venv/lib/python3.12/site-packages/scipy/optimize/_optimize.py:851: RuntimeWarning: invalid value encountered in subtract
  np.max(np.abs(fsim[0] - fsim[1:])) <= fatol):


Maximum likelihood result:
M  =  -19.2
H0  =  73.03398760350208
q0  =  -0.5280171275112104
j0  =  1.0
s0  =  -0.35
chi2min = inf
tempo total: 74.081 seg

 [-19.2         73.0339876   -0.52801713   1.          -0.35      ] 

       message: Maximum number of iterations has been exceeded.
       success: False
        status: 2
           fun: inf
             x: [-1.920e+01  7.303e+01 -5.280e-01  1.000e+00 -3.500e-01]
           nit: 10000
          nfev: 69999
 final_simplex: (array([[-1.920e+01,  7.303e+01, ...,  1.000e+00,
                        -3.500e-01],
                       [-1.920e+01,  7.303e+01, ...,  1.000e+00,
                        -3.500e-01],
                       ...,
                       [-1.920e+01,  7.303e+01, ...,  1.000e+00,
                        -3.500e-01],
                       [-1.920e+01,  7.303e+01, ...,  1.000e+00,
                        -3.500e-01]], shape=(6, 5)), array([       inf,        inf,        inf,        inf,
                           

In [16]:
info_PPHz31 = {
    "likelihood": {
        "lnlikehz": lnprobPPHz_31
    },

    "params": {
        "M": {"prior": {"min": -19.5, "max": 19.5}, "proposal": par_mlHzPP_31[0], "latex": "M"},
        "h0": {"prior": {"min": 40.0, "max": 100.0}, "proposal": par_mlHzPP_31[1], "latex": r"H_{0}"},
        "q0": {"prior": {"min": -2, "max": 0}, "proposal": par_mlHzPP_31[2], "latex": r"q_{0}"},
        "j0": {"prior": {"min": -5, "max": 5}, "proposal": par_mlHzPP_31[3], "latex": r"j_{0}"},
        "s0": {"prior": {"min": -10, "max": 10}, "proposal": par_mlHzPP_31[4], "latex": r"s_{0}"}
    },

    "sampler": {
        "polychord": {
            "nlive": 300
        }
    },

    "output": "chains/P31/PP/PPHz_31"
}

info_post_PPHz31 = {
    "output": "chains/P31/PP/PPHz_31",
    "post": {
        "skip_samples": 0.3,
        "suffix": "_post"
    }
}

In [17]:
_, sampler_PPHz_31 = run_cobaya(info_PPHz31, info_post_PPHz31)

----- INICIANDO SAMPLER -----
[output] Output to be read-from/written-into folder 'chains/P31/PP', with prefix 'PPHz_31'
[lnlikehz] Initialized external likelihood.
[polychord] `pypolychord` module loaded successfully from /home/nirk20/jupyter_venv/lib/python3.12/site-packages/pypolychord
[polychord] Storing raw PolyChord output in 'chains/P31/PP/PPHz_31_polychord_raw'.
[model] Measuring speeds... (this may take a few seconds)
[model] Setting measured speeds (per sec): {lnlikehz: 131.0}
[polychord] Parameter blocks and their oversampling factors:
[polychord] * 1 : ['M', 'h0', 'q0', 'j0', 's0']
[polychord] Calling PolyChord...
PolyChord: MPI is already initilised, not initialising, and will not finalize

PolyChord: Next Generation Nested Sampling
copyright: Will Handley, Mike Hobson & Anthony Lasenby
  version: 1.22.1
  release: 10th Jan 2024
    email: wh260@mrao.cam.ac.uk

Run Settings
nlive    :     300
nDims    :       5
nDerived :       2
Doing Clustering
Synchronous parallelisatio

#### Pantheon+SH0ES+CC

In [15]:
par_mlHzPS_31 = par0_22
resultHzPS_31 = find_bestfit(getpar_PSHz_31, parnames_22, par_mlHzPS_31)
par_mlHzPS_31 = resultHzPS_31['x']
print('\n',par_mlHzPS_31,'\n')
print(resultHzPS_31,'\n')

/home/nirk20/jupyter_venv/lib/python3.12/site-packages/scipy/optimize/_numdiff.py:596: RuntimeWarning: invalid value encountered in subtract
  df = fun(x1) - f0
/home/nirk20/jupyter_venv/lib/python3.12/site-packages/scipy/optimize/_optimize.py:851: RuntimeWarning: invalid value encountered in subtract
  np.max(np.abs(fsim[0] - fsim[1:])) <= fatol):


Maximum likelihood result:
M  =  -19.2
H0  =  73.03398760350208
q0  =  -0.5280171275112104
j0  =  1.0
s0  =  -0.35
chi2min = inf
tempo total: 73.668 seg

 [-19.2         73.0339876   -0.52801713   1.          -0.35      ] 

       message: Maximum number of iterations has been exceeded.
       success: False
        status: 2
           fun: inf
             x: [-1.920e+01  7.303e+01 -5.280e-01  1.000e+00 -3.500e-01]
           nit: 10000
          nfev: 69999
 final_simplex: (array([[-1.920e+01,  7.303e+01, ...,  1.000e+00,
                        -3.500e-01],
                       [-1.920e+01,  7.303e+01, ...,  1.000e+00,
                        -3.500e-01],
                       ...,
                       [-1.920e+01,  7.303e+01, ...,  1.000e+00,
                        -3.500e-01],
                       [-1.920e+01,  7.303e+01, ...,  1.000e+00,
                        -3.500e-01]], shape=(6, 5)), array([       inf,        inf,        inf,        inf,
                           

In [17]:
info_PSHz31 = {
    "likelihood": {
        "lnlikehz": lnprobPSHz_31
    },

    "params": {
        "M": {"prior": {"min": -19.5, "max": 19.5}, "proposal": par_mlHzPS_31[0], "latex": "M"},
        "h0": {"prior": {"min": 40.0, "max": 100.0}, "proposal": par_mlHzPS_31[1], "latex": r"H_{0}"},
        "q0": {"prior": {"min": -2, "max": 2}, "proposal": par_mlHzPS_31[2], "latex": r"q_{0}"},
        "j0": {"prior": {"min": -5, "max": 5}, "proposal": par_mlHzPS_31[3], "latex": r"j_{0}"},
        "s0": {"prior": {"min": -10, "max": 10}, "proposal": par_mlHzPS_31[4], "latex": r"s_{0}"}
    },

    "sampler": {
        "polychord": {
            "nlive": 300
        }
    },

    "output": "chains/P31/PS/PSHz_31"
}

info_post_PSHz31 = {
    "output": "chains/P31/PS/PSHz_31",
    "post": {
        "skip_samples": 0.3,
        "suffix": "_post"
    }
}

In [18]:
_, sampler_PSHz_31 = run_cobaya(info_PSHz31, info_post_PSHz31)

----- INICIANDO SAMPLER -----
[output] Output to be read-from/written-into folder 'chains/P31/PS', with prefix 'PSHz_31'
[lnlikehz] Initialized external likelihood.
[polychord] `pypolychord` module loaded successfully from /home/nirk20/jupyter_venv/lib/python3.12/site-packages/pypolychord
[polychord] Storing raw PolyChord output in 'chains/P31/PS/PSHz_31_polychord_raw'.
[model] Measuring speeds... (this may take a few seconds)
[model] Setting measured speeds (per sec): {lnlikehz: 189.0}
[polychord] Parameter blocks and their oversampling factors:
[polychord] * 1 : ['M', 'h0', 'q0', 'j0', 's0']
[polychord] Calling PolyChord...
PolyChord: MPI is already initilised, not initialising, and will not finalize

PolyChord: Next Generation Nested Sampling
copyright: Will Handley, Mike Hobson & Anthony Lasenby
  version: 1.22.1
  release: 10th Jan 2024
    email: wh260@mrao.cam.ac.uk

Run Settings
nlive    :     300
nDims    :       5
nDerived :       2
Doing Clustering
Synchronous parallelisatio